In [32]:
import torch
import torch.nn as nn

In [31]:
seed = 42
torch.manual_seed(seed)

In [30]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [29]:
class PatchEmbedding(nn.Module):
  def __init__(self, image_size, patch_size, d_model):
    super().__init__()
    self.image_size = image_size
    self.patch_size = patch_size
    self.d_model = d_model
    self.number_of_patches = (self.image_size//self.patch_size)**2
    self.projection = nn.Conv2d(in_channesls = 3, out_channels = self.d_model, kernel_size = self.patch_size, stride = self.patch_size)
    self.cls = nn.Parameter(torch.rand(1,d_model))
    self.positional_embedding = nn.Parameter(torch.rand(self.number_of_patches + 1, self.d_model))

  def forward(self, X):
    Batch = X.shape[0]
    emb = self.projection(X)
    cls = self.cls.expand(Batch, -1, -1)
    emb = torch.cat((cls,emb), dim = 1)
    emb += self.positional_embedding
    return emb

In [28]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, head):
    super().__init__()
    self.d_model = d_model
    self.head = head
    self.W_Q = nn.Parameter(torch.rand(self.d_model, self.d_model))
    self.W_K = nn.Parameter(torch.rand(self.d_model, self.d_model))
    self.W_V = nn.Parameter(torch.rand(self.d_model, self.d_model))
    self.projection = nn.Parameter(torch.rand(self.d_model, self.d_model))
    self.number_of_dim_head = self.d_model // self.head
    self.softmax = nn.Softmax(dim = -1)

  def forward(self, X):
    Q = X @ self.W_Q
    K = X @ self.W_K
    V = X @ self.W_V
    Q = Q.reshape(X.shape[0], X.shape[1], self.head, self.number_of_dim_head)
    K = K.reshape(X.shape[0], X.shape[1], self.head, self.number_of_dim_head)
    V = V.reshape(X.shape[0], X.shape[1], self.head, self.number_of_dim_head)
    Q = Q.transpose(2,1)
    K = K.transpose(2,1)
    V = V.transpose(2,1)
    A = (Q @ K.transpose(-1, -2))/self.d_model
    Attention = self.softmax(A) @ V
    Attention = Attention.transpose(1,2)
    Attention = Attention.reshape(X.shape[0], X.shape[1], self.d_model)
    Attention = Attention @ self.projection
    return Attention

In [27]:
class FeedForwardNeuralNetwork(nn.Module):
  def __init__(self, d_model):
    self.d_model = d_model
    self.Network = nn.Sequential(
        nn.Linear(self.d_model, 4 * self.d_model),
        nn.GELU(),
        nn.Dropout(p = 0.1),
        nn.Linear(4*self.d_model, self.d_model))

  def forward(self, X):
    return self.Network(X)


In [26]:
class VisionTransformer(nn.Module):
  def __init__(self, patch_size, image_size, d_model, head):
    super().__init__()
    self.patch_size = patch_size
    self.image_size = image_size
    self.d_model = d_model
    self.head = head
    self.PE = PatchEmbedding(self.image_size, self.patch_size, self.d_model)
    self.MHA = MultiHeadAttention(self.d_model, self.head)
    self.FFNN = FeedForwardNeuralNetwork(self.d_model)
    self.layer_norm1 = nn.LayerNorm(d_model)
    self.layer_norm2 = nn.LayerNorm(d_model)

  def forward(self, X):
    x = self.PE(X)
    x = x + self.MHA(self.layer_norm1(x))
    x = x + self.FFNN(self.layer_norm2(x))

    return x

